## Feature Engineering and Preliminary Feature Selection

Build a patient-level dataset from file-level features and run leakage-safe feature selection (train split only).

In [24]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import f_classif, mutual_info_classif

In [25]:
# Load features generated in 01_data_exploration (section 7.3)
processed_dir = Path('../..') / 'data' / 'processed'
parquet_path = processed_dir / 'timeseries_features.parquet'
csv_path = processed_dir / 'timeseries_features.csv'

if parquet_path.exists():
    df_features = pd.read_parquet(parquet_path)
elif csv_path.exists():
    df_features = pd.read_csv(csv_path)
else:
    raise FileNotFoundError('Feature table not found. Run section 7.3 in notebook 01 first.')

# Load patient demographics
base_path = Path('../..') / 'data' / 'raw' / 'pads-parkinsons-disease-smartwatch-dataset-1.0.0'
patients_path = base_path / 'patients'
questionnaire_path = base_path / 'questionnaire'

patient_files = sorted(list(patients_path.glob('*.json')))
questionnaire_files = sorted(list(questionnaire_path.glob('*.json')))

import json
patient_rows = []
for fp in patient_files:
    with open(fp, 'r') as f:
        patient_rows.append(json.load(f))
df_patients = pd.DataFrame(patient_rows)

# Load questionnaire items (subject-level yes/no responses)
questionnaire_rows = []
for fp in questionnaire_files:
    with open(fp, 'r') as f:
        q = json.load(f)
    subject_id = q.get('subject_id')
    questionnaire_name = q.get('questionnaire_name')
    for item in q.get('item', []):
        questionnaire_rows.append({
            'subject_id': subject_id,
            'questionnaire_name': questionnaire_name,
            'link_id': item.get('link_id'),
            'answer': item.get('answer')
        })

df_questionnaire_items = pd.DataFrame(questionnaire_rows)
if len(df_questionnaire_items) > 0:
    df_questionnaire_items['answer_num'] = df_questionnaire_items['answer'].map({True: 1, False: 0})

print(f'df_features shape: {df_features.shape}')
print(f'df_patients shape: {df_patients.shape}')
print(f'df_questionnaire_items shape: {df_questionnaire_items.shape}')
print(df_features.head())
print(df_patients.head())
print(df_questionnaire_items.head())

df_features shape: (10318, 33)
df_patients shape: (469, 14)
df_questionnaire_items shape: (14070, 5)
   subject_id condition  record_name device_location  \
0           1   Healthy      Relaxed       LeftWrist   
1           1   Healthy      Relaxed      RightWrist   
2           1   Healthy  RelaxedTask       LeftWrist   
3           1   Healthy  RelaxedTask      RightWrist   
4           1   Healthy  StretchHold       LeftWrist   

                                   file_name  rows_expected  n_rows_raw  \
0       timeseries/001_Relaxed_LeftWrist.txt           2048        2048   
1      timeseries/001_Relaxed_RightWrist.txt           2048        2048   
2   timeseries/001_RelaxedTask_LeftWrist.txt           2048        2048   
3  timeseries/001_RelaxedTask_RightWrist.txt           2048        2048   
4   timeseries/001_StretchHold_LeftWrist.txt           1024        1024   

   n_rows_used  row_coverage  sampling_hz  ...  acc_mag_mean  acc_mag_std  \
0         1997           1.0   100

In [31]:
# 1) Build patient-level table from file-level sensor features
feature_exclude = {'rows_expected'}
feature_cols_file = [
    c for c in df_features.columns
    if c not in ['subject_id', 'condition', 'record_name', 'device_location', 'file_name']
    and c not in feature_exclude
    and np.issubdtype(df_features[c].dtype, np.number)
]

patient_agg = (
    df_features
    .groupby(['subject_id', 'condition'], as_index=False)[feature_cols_file]
    .agg(['mean', 'std'])
)

patient_agg.columns = [
    col if isinstance(col, str) else f'{col[0]}_{col[1]}'
    for col in patient_agg.columns
]
patient_agg = patient_agg.rename(columns={'subject_id_': 'subject_id', 'condition_': 'condition'})

def normalize_subject_id(series):
    s = series.astype(str).str.strip()
    s = s.str.replace(r'\.0$', '', regex=True)
    s = s.str.replace(r'^(patient_|subject_)', '', regex=True)
    s = s.str.extract(r'(\d+)', expand=False).fillna(s)
    s = s.apply(lambda x: x.zfill(3) if isinstance(x, str) and x.isdigit() and len(x) < 3 else x)
    return s

patient_agg['subject_id'] = normalize_subject_id(patient_agg['subject_id'])

# 2) Merge demographics (fixed columns known from dataset)
demo_cols = ['age', 'gender', 'height', 'weight']
df_demo = df_patients[['id'] + demo_cols].copy().rename(columns={'id': 'subject_id'})
df_demo['subject_id'] = normalize_subject_id(df_demo['subject_id'])
patient_df = patient_agg.merge(df_demo, on='subject_id', how='left')

# 3) Add questionnaire features (subject-level)
#    - one column per question
if len(df_questionnaire_items) > 0 and 'answer_num' in df_questionnaire_items.columns:
    q_work = df_questionnaire_items.copy()
    q_work['subject_id'] = normalize_subject_id(q_work['subject_id'])

    q_features = (
        q_work
        .pivot(index='subject_id', columns='link_id', values='answer_num')
        .add_prefix('q_')
        .reset_index()
    )

    patient_df['subject_id'] = normalize_subject_id(patient_df['subject_id'])
    patient_df = patient_df.merge(q_features, on='subject_id', how='left')

# 4) Define binary target y = PD vs non-PD
condition_norm = patient_df['condition'].astype(str).str.lower().str.strip()
y = (condition_norm == "parkinson's disease").astype(int)

if y.sum() == 0:
    y = condition_norm.str.contains('parkinson', regex=False).astype(int)
    y = np.where(condition_norm.str.contains('atypical|secondary|parkinsonism', regex=True), 0, y)
    y = pd.Series(y, index=patient_df.index)

patient_df['y_pd_binary'] = y

print(f'Patient-level rows: {len(patient_df)}')
print('Target distribution (0=non-PD, 1=PD):')
print(patient_df['y_pd_binary'].value_counts())

# Controlling the merged dataset quality
print('\nFirst 5 rows of merged patient_df:')
display(patient_df.head())

patient_missing = patient_df.isna().sum()
total_missing = int(patient_missing.sum())
cols_with_missing = int((patient_missing > 0).sum())
print('\nMerged patient_df missing report:')
print(f'Total missing values: {total_missing}')
print(f'Columns with at least 1 missing: {cols_with_missing}/{patient_df.shape[1]}')

top_missing_patient_df = patient_missing[patient_missing > 0].sort_values(ascending=False)
if len(top_missing_patient_df) > 0:
    print('Top columns by missing count:')
    print(top_missing_patient_df.head(15).to_string())
else:
    print('No missing values found in patient_df.')

Patient-level rows: 469
Target distribution (0=non-PD, 1=PD):
y_pd_binary
1    276
0    193
Name: count, dtype: int64

First 5 rows of merged patient_df:


,subject_id,condition,n_rows_raw_mean,n_rows_raw_std,n_rows_used_mean,n_rows_used_std,row_coverage_mean,row_coverage_std,sampling_hz_mean,sampling_hz_std,...,q_22,q_23,q_24,q_25,q_26,q_27,q_28,q_29,q_30,y_pd_binary
0,001,Healthy,1303.272727,466.782521,1252.772727,466.782802,1.0,0.0,99.696039,0.363960,...,0,0,0,0,0,0,0,0,0,0
1,002,Other Movement Disorders,1303.272727,466.782521,1252.772727,466.782802,1.0,0.0,99.693372,0.356011,...,1,1,0,1,0,1,0,1,0,0
2,003,Healthy,1303.272727,466.782521,1252.818182,466.858767,1.0,0.0,99.709051,0.379732,...,0,0,0,0,0,0,0,0,0,0
3,004,Parkinson's,1303.272727,466.782521,1252.818182,466.754308,1.0,0.0,99.687758,0.365762,...,1,1,0,0,1,1,1,0,0,1
4,005,Parkinson's,1303.272727,466.782521,1252.772727,466.782802,1.0,0.0,99.704164,0.378040,...,1,1,1,1,1,1,0,0,0,1



Merged patient_df missing report:
Total missing values: 0
Columns with at least 1 missing: 0/91
No missing values found in patient_df.


In [33]:
# 3) Train/test split (patient-level, stratified)
X_all = patient_df.drop(columns=['subject_id', 'condition', 'y_pd_binary'], errors='ignore')
X_all = X_all.select_dtypes(include=[np.number])
y_all = patient_df['y_pd_binary']

questionnaire_feature_cols = [c for c in X_all.columns if c.startswith('q_')]
sensor_feature_cols = [c for c in X_all.columns if not c.startswith('q_')]

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, random_state=42, stratify=y_all
)

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(y_train.value_counts(normalize=True).round(3))
print(f'Total numeric features: {X_all.shape[1]}')
print(f'Sensor/demographic features: {len(sensor_feature_cols)}')
print(f'Questionnaire-derived features: {len(questionnaire_feature_cols)}')

print('\nFirst 5 rows of X_train:')
display(X_train.head())
print('\nFirst 5 rows of X_test:')
display(X_test.head())

X_train: (375, 87) | X_test: (94, 87)
y_pd_binary
1    0.589
0    0.411
Name: proportion, dtype: float64
Total numeric features: 87
Sensor/demographic features: 57
Questionnaire-derived features: 30

First 5 rows of X_train:


,n_rows_raw_mean,n_rows_raw_std,n_rows_used_mean,n_rows_used_std,row_coverage_mean,row_coverage_std,sampling_hz_mean,sampling_hz_std,missing_ratio_before_fill_mean,missing_ratio_before_fill_std,...,q_21,q_22,q_23,q_24,q_25,q_26,q_27,q_28,q_29,q_30
148,1303.272727,466.782521,1252.772727,466.782802,1.0,0.0,99.698344,0.366270,0.0,0.0,...,0,0,0,0,1,0,0,0,0,0
287,1303.272727,466.782521,1252.863636,466.726115,1.0,0.0,99.726193,0.384581,0.0,0.0,...,0,1,1,0,0,1,0,0,1,0
294,1303.272727,466.782521,1252.818182,466.754308,1.0,0.0,99.693834,0.360631,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
208,1303.272727,466.782521,1252.772727,466.782802,1.0,0.0,99.689732,0.370770,0.0,0.0,...,0,0,0,0,0,1,0,1,0,0
75,1303.272727,466.782521,1252.818182,466.754308,1.0,0.0,99.699226,0.365302,0.0,0.0,...,0,0,1,0,1,0,0,1,0,0



First 5 rows of X_test:


,n_rows_raw_mean,n_rows_raw_std,n_rows_used_mean,n_rows_used_std,row_coverage_mean,row_coverage_std,sampling_hz_mean,sampling_hz_std,missing_ratio_before_fill_mean,missing_ratio_before_fill_std,...,q_21,q_22,q_23,q_24,q_25,q_26,q_27,q_28,q_29,q_30
107,1303.272727,466.782521,1252.772727,466.782802,1.0,0.0,100.144191,0.622356,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
416,1303.272727,466.782521,1252.772727,466.782802,1.0,0.0,99.701654,0.362093,0.0,0.0,...,1,1,0,0,1,1,0,0,0,0
258,1303.272727,466.782521,1252.772727,466.782802,1.0,0.0,99.684202,0.369181,0.0,0.0,...,0,0,1,1,0,0,1,0,1,0
452,1303.272727,466.782521,1252.454545,466.564315,1.0,0.0,100.146833,0.625202,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
46,1303.272727,466.782521,1252.772727,466.782802,1.0,0.0,99.702725,0.365012,0.0,0.0,...,0,0,0,1,1,0,1,0,0,0


In [ ]:
# 4) Build A/B/C feature setups with identical train-only selection pipeline

def run_train_only_feature_selection(X_train_in, X_test_in, y_train_in, missing_threshold=0.20, corr_threshold=0.95, top_k=25):
    # Missing filter (fit on train)
    train_missing_ratio = X_train_in.isna().mean()
    keep_missing = train_missing_ratio[train_missing_ratio <= missing_threshold].index.tolist()
    dropped_missing = [c for c in X_train_in.columns if c not in keep_missing]

    X_train_w = X_train_in[keep_missing].copy()
    X_test_w = X_test_in[keep_missing].copy()

    # Median imputation (fit on train)
    train_medians = X_train_w.median()
    X_train_w = X_train_w.fillna(train_medians)
    X_test_w = X_test_w.fillna(train_medians)

    # Correlation filter (fit on train)
    corr_matrix = X_train_w.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop_corr = [col for col in upper.columns if any(upper[col] > corr_threshold)]

    X_train_f = X_train_w.drop(columns=to_drop_corr)
    X_test_f = X_test_w.drop(columns=to_drop_corr)

    # Ranking on train only
    f_vals, p_vals = f_classif(X_train_f, y_train_in)
    mi_vals = mutual_info_classif(X_train_f, y_train_in, random_state=42)

    fs_df_local = pd.DataFrame({
        'feature': X_train_f.columns,
        'anova_f': f_vals,
        'anova_p': p_vals,
        'mutual_info': mi_vals
    })

    fs_df_local['rank_anova'] = fs_df_local['anova_p'].rank(ascending=True, method='average')
    fs_df_local['rank_mi'] = fs_df_local['mutual_info'].rank(ascending=False, method='average')
    fs_df_local['rank_combined'] = (fs_df_local['rank_anova'] + fs_df_local['rank_mi']) / 2
    fs_df_local = fs_df_local.sort_values('rank_combined', ascending=True).reset_index(drop=True)

    k_local = min(top_k, len(fs_df_local))
    selected_local = fs_df_local.head(k_local)['feature'].tolist()

    X_train_sel = X_train_f[selected_local].copy()
    X_test_sel = X_test_f[selected_local].copy()

    info = {
        'n_input_features': int(X_train_in.shape[1]),
        'n_after_missing': int(X_train_w.shape[1]),
        'n_removed_missing': int(len(dropped_missing)),
        'n_after_corr': int(X_train_f.shape[1]),
        'n_removed_corr': int(len(to_drop_corr)),
        'n_selected': int(len(selected_local))
    }

    return X_train_sel, X_test_sel, fs_df_local, selected_local, info


# Define setup column groups
all_cols = X_train.columns.tolist()
q_cols = [c for c in all_cols if c.startswith('q_')]
demo_candidates = ['age', 'gender', 'height', 'weight']
demo_cols_num = [c for c in demo_candidates if c in all_cols]
sensor_cols_only = [c for c in all_cols if c not in q_cols and c not in demo_cols_num]

setups = {
    'A_sensor_demo': sensor_cols_only + demo_cols_num,
    'B_questionnaire_demo': q_cols + demo_cols_num,
    'C_all_modalities': all_cols
}

setup_results = {}
setup_summary_rows = []

for setup_name, cols in setups.items():
    cols = list(dict.fromkeys(cols))
    Xtr = X_train[cols].copy()
    Xte = X_test[cols].copy()

    Xtr_sel, Xte_sel, fs_table, feat_list, info = run_train_only_feature_selection(
        Xtr, Xte, y_train, missing_threshold=0.20, corr_threshold=0.95, top_k=25
    )

    setup_results[setup_name] = {
        'X_train_selected': Xtr_sel,
        'X_test_selected': Xte_sel,
        'feature_selection_table': fs_table,
        'final_feature_list': feat_list,
        'info': info
    }

    setup_summary_rows.append({
        'setup': setup_name,
        **info
    })

setup_summary_df = pd.DataFrame(setup_summary_rows).sort_values('setup').reset_index(drop=True)
print('A/B/C setup summary (same patient split, train-only feature selection):')
print(setup_summary_df)

for setup_name in sorted(setup_results.keys()):
    print(f"\n{setup_name}: selected shape train/test -> "
          f"{setup_results[setup_name]['X_train_selected'].shape} / "
          f"{setup_results[setup_name]['X_test_selected'].shape}")

c:\Users\paolo\Desktop\Behavioral-Health-Informatic-Project\bhi-env\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [4 5 8 9] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Users\paolo\Desktop\Behavioral-Health-Informatic-Project\bhi-env\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
c:\Users\paolo\Desktop\Behavioral-Health-Informatic-Project\bhi-env\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [4 5 8 9] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Users\paolo\Desktop\Behavioral-Health-Informatic-Project\bhi-env\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


A/B/C setup summary (same patient split, train-only feature selection):
                  setup  n_input_features  n_after_missing  n_removed_missing  \
0         A_sensor_demo                57               57                  0   
1  B_questionnaire_demo                33               33                  0   
2      C_all_modalities                87               87                  0   

   n_after_corr  n_removed_corr  n_selected  
0            42              15          25  
1            33               0          25  
2            72              15          25  

A_sensor_demo: selected shape train/test -> (375, 25) / (94, 25)

B_questionnaire_demo: selected shape train/test -> (375, 25) / (94, 25)

C_all_modalities: selected shape train/test -> (375, 25) / (94, 25)


In [ ]:
# 5) Save A/B/C selected datasets for downstream modeling
final_dir = Path('../..') / 'data' / 'final'
final_dir.mkdir(parents=True, exist_ok=True)

for setup_name, payload in setup_results.items():
    setup_slug = setup_name.lower()

    payload['X_train_selected'].to_csv(final_dir / f'X_train_{setup_slug}_selected.csv', index=False)
    payload['X_test_selected'].to_csv(final_dir / f'X_test_{setup_slug}_selected.csv', index=False)

    payload['feature_selection_table'].to_csv(
        final_dir / f'feature_selection_table_{setup_slug}.csv', index=False
    )
    pd.Series(payload['final_feature_list'], name='feature').to_csv(
        final_dir / f'final_feature_list_{setup_slug}.csv', index=False
    )

setup_summary_df.to_csv(final_dir / 'setup_summary_abc.csv', index=False)

print(f"Saved A/B/C selected artifacts in: {final_dir.resolve()}")
print('Generated files:')
for setup_name in sorted(setup_results.keys()):
    setup_slug = setup_name.lower()
    print(f"- X_train_{setup_slug}_selected.csv")
    print(f"- X_test_{setup_slug}_selected.csv")
    print(f"- feature_selection_table_{setup_slug}.csv")
    print(f"- final_feature_list_{setup_slug}.csv")
print('- setup_summary_abc.csv')